# APD Vision — training full-PPE Safety RD v1

Notebook Google Colab ini melatih satu dataset: `akfa-beqxl/safety-rd-v1`, lalu menormalkan label menjadi empat kelas proyek dalam urutan `helmet`, `vest`, `gloves`, `shoes`.

Pengamanan: ZIP tool minimal diunggah manual lewat panel Files tanpa akses repository privat; API key hanya dari Colab Secrets; audit kelas/split/label/duplikat sebelum training; gold reviewed tidak pernah masuk training; output selalu bernama **candidate** dan tidak mengganti model aktif.

Aktifkan **Runtime → Change runtime type → T4 GPU**, lalu jalankan sel secara berurutan.

## 1. Periksa GPU dan pasang dependensi

In [ ]:
import subprocess, sys
subprocess.run(['nvidia-smi'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics>=8.3,<9', 'roboflow>=1.1', 'pyyaml>=6.0'], check=True)

## 2. Siapkan tool dari upload manual

Sebelum menjalankan sel berikut, buka panel **Files** di kiri Colab, klik **Upload**, lalu unggah `ppe-colab-tools.zip` ke `/content`. ZIP kecil ini cukup memuat `training/fetch_datasets.py`, `training/merge_datasets.py`, dan `training/evaluate_model.py`; seluruh repository tidak diperlukan.

In [ ]:
import os, shutil, stat, sys, zipfile
from pathlib import Path

TOOLS_ZIP_PATH = '/content/ppe-colab-tools.zip'
EXTRACT_ROOT = Path('/content/colab_tools').resolve()
if TOOLS_ZIP_PATH:
    tools_zip = Path(TOOLS_ZIP_PATH).expanduser().resolve()
    zip_candidates = [tools_zip]
else:
    zip_candidates = sorted(Path('/content').glob('*.zip'))
if len(zip_candidates) != 1 or not zip_candidates[0].is_file():
    raise RuntimeError(f'Unggah ppe-colab-tools.zip ke /content atau sesuaikan TOOLS_ZIP_PATH. Ditemukan: {zip_candidates}')
tools_zip = zip_candidates[0].resolve()
if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
EXTRACT_ROOT.mkdir(parents=True)
with zipfile.ZipFile(tools_zip) as archive:
    for member in archive.infolist():
        destination = (EXTRACT_ROOT / member.filename).resolve()
        if destination != EXTRACT_ROOT and EXTRACT_ROOT not in destination.parents:
            raise RuntimeError(f'Path ZIP tidak aman: {member.filename}')
        mode = member.external_attr >> 16
        if stat.S_ISLNK(mode):
            raise RuntimeError(f'Symlink dalam ZIP tidak diizinkan: {member.filename}')
    archive.extractall(EXTRACT_ROOT)
project_candidates = sorted({p.parent.parent.resolve() for p in EXTRACT_ROOT.rglob('training/merge_datasets.py') if (p.parent / 'fetch_datasets.py').is_file() and (p.parent / 'evaluate_model.py').is_file()})
if len(project_candidates) != 1:
    raise RuntimeError(f'ZIP tool harus memuat training/fetch_datasets.py, training/merge_datasets.py, dan training/evaluate_model.py. Ditemukan: {project_candidates}')
PROJECT_DIR = project_candidates[0]
TRAINING_DIR = PROJECT_DIR / 'training'
for tool in ('fetch_datasets.py', 'merge_datasets.py', 'evaluate_model.py'):
    compile((TRAINING_DIR / tool).read_text(), tool, 'exec')
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print('Tool training siap:', TRAINING_DIR)

## 3. Muat secret dan Google Drive

Buat Colab Secret bernama `ROBOFLOW_API_KEY` dan aktifkan akses notebook. Nilainya tidak dicetak atau ditulis ke file.

In [ ]:
from google.colab import drive, userdata
roboflow_api_key = userdata.get('ROBOFLOW_API_KEY')
if not roboflow_api_key:
    raise RuntimeError('Secret ROBOFLOW_API_KEY belum tersedia atau akses notebook belum diaktifkan.')
os.environ['ROBOFLOW_API_KEY'] = roboflow_api_key
drive.mount('/content/drive')
print('Secret termuat tanpa menampilkan nilainya; Google Drive siap.')

## 4. Konfigurasi

Biarkan `LICENSE_CONFIRMED = False` sampai lisensi pada halaman Universe atau README hasil unduhan telah diperiksa dan sesuai dengan tujuan penggunaan.

In [ ]:
from datetime import datetime, timezone
DATASET_SPEC = 'akfa-beqxl/safety-rd-v1'
TARGET_CLASSES = ['helmet', 'vest', 'gloves', 'shoes']
SOURCE_DIR = Path('/content/datasets/safety-rd-v1')
MERGED_DIR = Path('/content/datasets/safety-rd-full4')
DRIVE_ROOT = Path('/content/drive/MyDrive/PPE_Guard')
RUN_ID = datetime.now(timezone.utc).strftime('safety_rd_v1_%Y%m%d_%H%M%S_utc')
RUNS_DIR = DRIVE_ROOT / 'runs'
AUDIT_DIR = DRIVE_ROOT / 'audits' / RUN_ID
LICENSE_CONFIRMED = False
RUN_GOLD_EVALUATION = False
GOLD_ZIP_PATH = ''  # Opsional: ZIP folder datasets/gold di Google Drive.
EPOCHS, IMGSZ, BATCH, PATIENCE, SEED = 150, 960, -1, 30, 42
MIN_RECOMMENDED_INSTANCES = 2000
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
print('Run ID:', RUN_ID, '| dataset training tunggal:', DATASET_SPEC)

## 5. Unduh dataset

Spec tanpa nomor versi mengambil versi terbaru. Catat versi yang tercetak agar eksperimen dapat direproduksi.

In [ ]:
import shutil
from roboflow import Roboflow
from training.fetch_datasets import fetch
if SOURCE_DIR.exists(): shutil.rmtree(SOURCE_DIR)
rf = Roboflow(api_key=roboflow_api_key)
SOURCE_DIR = Path(fetch(DATASET_SPEC, str(SOURCE_DIR), rf)).resolve()
del rf, roboflow_api_key
os.environ.pop('ROBOFLOW_API_KEY', None)
print('Dataset tersedia di:', SOURCE_DIR)

## 6. Audit sebelum training

Audit menghentikan proses jika empat kelas positif tidak lengkap, split train/valid/test kosong, label rusak/hilang, atau gambar identik muncul pada split berbeda. Bbox kecil didefinisikan sebagai area relatif `<1%`; sangat kecil `<0,25%`.

In [ ]:
import hashlib, json
from collections import Counter, defaultdict
import yaml
from training.merge_datasets import CLASS_MAP
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png'}
data_yaml = SOURCE_DIR / 'data.yaml'
raw_names = yaml.safe_load(data_yaml.read_text())['names']
names = [str(raw_names[k]) for k in sorted(raw_names, key=lambda x: int(x))] if isinstance(raw_names, dict) else list(map(str, raw_names))
normalized = [name.strip().lower() for name in names]
source_to_target = {i: CLASS_MAP[name] for i, name in enumerate(normalized) if name in CLASS_MAP}
missing_classes = sorted(set(TARGET_CLASSES) - set(source_to_target.values()))
if missing_classes:
    raise RuntimeError(f'Kelas target hilang: {missing_classes}; kelas sumber: {names}')

def split_dirs(root):
    result = {}
    for source_split, split in [('train','train'), ('valid','val'), ('val','val'), ('test','test')]:
        for images, labels in [(root/source_split/'images', root/source_split/'labels'), (root/'images'/source_split, root/'labels'/source_split)]:
            if images.exists():
                if split in result: raise RuntimeError(f'Split {split} ditemukan lebih dari sekali.')
                result[split] = (images, labels)
    return result

splits = split_dirs(SOURCE_DIR)
if set(splits) != {'train', 'val', 'test'}:
    raise RuntimeError(f'Split train/valid/test wajib tersedia. Ditemukan: {sorted(splits)}')
split_images, split_instances = Counter(), defaultdict(Counter)
totals, small, very_small, background = Counter(), Counter(), Counter(), Counter()
missing_labels, invalid_labels, hashes = [], [], defaultdict(list)
for split, (image_dir, label_dir) in splits.items():
    images = sorted(p for p in image_dir.rglob('*') if p.suffix.lower() in IMAGE_SUFFIXES)
    split_images[split] = len(images)
    for image in images:
        hashes[hashlib.sha256(image.read_bytes()).hexdigest()].append((split, str(image)))
        label = label_dir / f'{image.stem}.txt'
        if not label.exists():
            missing_labels.append(str(label)); continue
        mapped = 0
        for line_no, line in enumerate(label.read_text().splitlines(), 1):
            if not line.strip(): continue
            try:
                parts = line.split()
                if len(parts) < 5: raise ValueError('kolom kurang dari lima')
                class_id = int(parts[0]); x, y, w, h = map(float, parts[1:5])
                if class_id < 0 or class_id >= len(names): raise ValueError('ID kelas di luar data.yaml')
                if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1): raise ValueError('bbox di luar rentang YOLO')
            except ValueError as error:
                invalid_labels.append(f'{label}:{line_no}: {error}'); continue
            if class_id not in source_to_target: continue
            target, area = source_to_target[class_id], w*h
            totals[target] += 1; split_instances[split][target] += 1; mapped += 1
            small[target] += area < .01; very_small[target] += area < .0025
        if mapped == 0: background[split] += 1
cross_split = [{'sha256': h, 'locations': loc} for h, loc in hashes.items() if len({x[0] for x in loc}) > 1]
same_split_duplicates = sum(len(loc) > 1 and len({x[0] for x in loc}) == 1 for loc in hashes.values())
license_files = sorted(p for p in SOURCE_DIR.rglob('*') if p.is_file() and ('license' in p.name.lower() or p.name.lower().startswith('readme')))
audit = {
    'dataset_spec': DATASET_SPEC, 'source_classes': names,
    'class_mapping': {names[i]: target for i, target in source_to_target.items()},
    'split_images': dict(split_images),
    'split_instances': {s: dict(v) for s, v in split_instances.items()},
    'total_instances': dict(totals), 'background_images': dict(background),
    'missing_label_count': len(missing_labels), 'invalid_label_count': len(invalid_labels),
    'cross_split_duplicate_groups': len(cross_split), 'same_split_duplicate_groups': same_split_duplicates,
    'small_bbox_percent': {c: round(100*small[c]/totals[c], 2) if totals[c] else 0.0 for c in TARGET_CLASSES},
    'very_small_bbox_percent': {c: round(100*very_small[c]/totals[c], 2) if totals[c] else 0.0 for c in TARGET_CLASSES},
    'license_files': [str(p.relative_to(SOURCE_DIR)) for p in license_files]
}
audit_path = AUDIT_DIR / 'dataset_audit.json'
audit_path.write_text(json.dumps(audit, indent=2, ensure_ascii=False))
print(json.dumps(audit, indent=2, ensure_ascii=False))
for c in TARGET_CLASSES:
    if totals[c] < MIN_RECOMMENDED_INSTANCES: print(f'PERINGATAN: {c} hanya {totals[c]} instance; target awal {MIN_RECOMMENDED_INSTANCES}.')
if missing_labels: raise RuntimeError(f'{len(missing_labels)} label hilang. Contoh: {missing_labels[:5]}')
if invalid_labels: raise RuntimeError(f'{len(invalid_labels)} label rusak. Contoh: {invalid_labels[:5]}')
if cross_split: raise RuntimeError(f'{len(cross_split)} grup gambar identik muncul pada split berbeda.')
if any(totals[c] == 0 for c in TARGET_CLASSES): raise RuntimeError(f'Kelas target kosong: {dict(totals)}')
print('AUDIT STRUKTURAL LOLOS. Tinjau sampel anotasi dan lisensi sebelum melanjutkan.')

## 7. Konfirmasi lisensi dan remap empat kelas

Periksa file lisensi/README yang dilaporkan dan sampel Roboflow Health Check. Pastikan APD yang terlihat dilabeli konsisten, khususnya tangan dan kaki pekerja full-body. Ubah `LICENSE_CONFIRMED = True` hanya setelah pemeriksaan.

In [ ]:
if not LICENSE_CONFIRMED:
    raise RuntimeError('Lisensi belum dikonfirmasi. Tinjau Universe/README lalu ubah LICENSE_CONFIRMED menjadi True.')
source_config = yaml.safe_load(data_yaml.read_text())
source_config['names'] = {i: name for i, name in enumerate(normalized)}
data_yaml.write_text(yaml.safe_dump(source_config, sort_keys=False))
if MERGED_DIR.exists(): shutil.rmtree(MERGED_DIR)
subprocess.run([sys.executable, str(TRAINING_DIR / 'merge_datasets.py'), '--src', str(SOURCE_DIR), '--out', str(MERGED_DIR)], check=True)
merged = yaml.safe_load((MERGED_DIR/'data.yaml').read_text())
if list(merged['names'].values()) != TARGET_CLASSES: raise RuntimeError(f'Urutan kelas salah: {merged["names"]}')
merged_counts = Counter()
for label in (MERGED_DIR/'labels').rglob('*.txt'):
    for line in label.read_text().splitlines():
        if line.strip(): merged_counts[TARGET_CLASSES[int(line.split()[0])]] += 1
if any(merged_counts[c] != totals[c] for c in TARGET_CLASSES): raise RuntimeError(f'Jumlah berubah saat remap: {dict(merged_counts)}')
print('REMAP LOLOS:', dict(merged_counts))

## 8. Latih YOLO11s

Checkpoint disimpan langsung ke Google Drive. Baseline ini tidak memakai copy-paste atau dataset specialist agar kualitas dataset full-PPE dapat diukur tanpa confounder.

In [ ]:
import torch
from ultralytics import YOLO
if not torch.cuda.is_available(): raise RuntimeError('GPU CUDA tidak aktif. Ubah runtime Colab ke T4 GPU.')
run_config = {'dataset_spec': DATASET_SPEC, 'run_id': RUN_ID, 'classes': TARGET_CLASSES, 'weights': 'yolo11s.pt', 'epochs': EPOCHS, 'imgsz': IMGSZ, 'batch': BATCH, 'patience': PATIENCE, 'seed': SEED}
(AUDIT_DIR/'run_config.json').write_text(json.dumps(run_config, indent=2))
model = YOLO('yolo11s.pt')
result = model.train(data=str(MERGED_DIR/'data.yaml'), imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH, patience=PATIENCE, device=0, workers=2, project=str(RUNS_DIR), name=RUN_ID, seed=SEED, deterministic=True, cos_lr=True, close_mosaic=10, save_period=10, plots=True)
RUN_DIR = Path(result.save_dir); CANDIDATE = RUN_DIR/'weights'/'best.pt'
if not CANDIDATE.exists(): raise RuntimeError(f'Checkpoint tidak ditemukan: {CANDIDATE}')
print('Training selesai. Candidate:', CANDIDATE)

## 9. Evaluasi internal

Metrik test dari sumber yang sama hanya untuk diagnosis dan **bukan** dasar promosi.

In [ ]:
internal_metrics = RUN_DIR/'metrics_internal.json'
subprocess.run([sys.executable, str(TRAINING_DIR / 'evaluate_model.py'), '--weights', str(CANDIDATE), '--data', str(MERGED_DIR/'data.yaml'), '--split', 'test', '--imgsz', str(IMGSZ), '--device', '0', '--json-out', str(internal_metrics)], check=True)
print('Status tetap CANDIDATE; belum ada promosi.')

## 10. Evaluasi opsional pada gold reviewed 340 gambar

ZIP folder lokal `datasets/gold` harus memuat `data.yaml`, `manifest.jsonl`, `images/test`, dan `labels/test`. Isi `GOLD_ZIP_PATH` atau biarkan kosong untuk upload, lalu set `RUN_GOLD_EVALUATION = True`. Sel memverifikasi 340 hash, menolak path traversal, dan menolak overlap dengan training. Gate hanya mengevaluasi; tidak mempromosikan.

In [ ]:
if RUN_GOLD_EVALUATION:
    import stat, zipfile
    from google.colab import files
    if GOLD_ZIP_PATH:
        gold_zip = Path(GOLD_ZIP_PATH).expanduser().resolve()
    else:
        uploaded = files.upload()
        if len(uploaded) != 1: raise RuntimeError('Unggah tepat satu ZIP gold.')
        name, content = next(iter(uploaded.items()))
        if not name.lower().endswith('.zip'): raise RuntimeError('Gold harus ZIP.')
        gold_zip = (Path('/content')/Path(name).name).resolve(); gold_zip.write_bytes(content)
    gold_root = Path('/content/gold_reviewed').resolve()
    if gold_root.exists(): shutil.rmtree(gold_root)
    gold_root.mkdir()
    with zipfile.ZipFile(gold_zip) as archive:
        for member in archive.infolist():
            destination = (gold_root/member.filename).resolve()
            if destination != gold_root and gold_root not in destination.parents: raise RuntimeError(f'Path ZIP tidak aman: {member.filename}')
            mode = member.external_attr >> 16
            if stat.S_ISLNK(mode): raise RuntimeError(f'Symlink dalam ZIP tidak diizinkan: {member.filename}')
        archive.extractall(gold_root)
    manifests = list(gold_root.rglob('manifest.jsonl'))
    if len(manifests) != 1: raise RuntimeError(f'Harus ada satu manifest; ditemukan {len(manifests)}.')
    GOLD_DIR = manifests[0].parent.resolve()
    records = [json.loads(line) for line in manifests[0].read_text().splitlines() if line.strip()]
    expected = {row['sha256'] for row in records}
    gold_images = [p for p in GOLD_DIR.rglob('*') if p.suffix.lower() in IMAGE_SUFFIXES]
    actual = {hashlib.sha256(p.read_bytes()).hexdigest() for p in gold_images}
    if len(records) != 340 or len(gold_images) != 340 or actual != expected: raise RuntimeError('Gold bukan set aktif 340 gambar atau gagal cek manifest.')
    if actual & set(hashes): raise RuntimeError(f'DATA LEAKAGE: {len(actual & set(hashes))} gambar gold ada di training.')
    gold_yaml = GOLD_DIR/'data.yaml'; config = yaml.safe_load(gold_yaml.read_text()); config['path'] = str(GOLD_DIR); gold_yaml.write_text(yaml.safe_dump(config, sort_keys=False))
    gold_metrics = RUN_DIR/'metrics_gold_reviewed_340.json'
    command = [sys.executable, str(TRAINING_DIR / 'evaluate_model.py'), '--weights', str(CANDIDATE), '--data', str(gold_yaml), '--split', 'test', '--imgsz', str(IMGSZ), '--device', '0', '--json-out', str(gold_metrics), '--min-map', '0.45', '--min-class-map', 'helmet=0.6', 'vest=0.6', 'gloves=0.7', 'shoes=0.6']
    gate = subprocess.run(command, check=False)
    print('GATE GOLD LOLOS; tetap review sebelum promosi manual.' if gate.returncode == 0 else 'GATE GOLD GAGAL; candidate tidak boleh dipromosikan.')
else:
    print('Evaluasi gold dilewati. Status tetap CANDIDATE dan tidak boleh dipromosikan.')

## 11. Unduh artefak

Nama file diawali `candidate_` agar tidak tertukar dengan bobot aktif. Run lengkap tetap ada di `MyDrive/PPE_Guard/runs`.

In [ ]:
from google.colab import files
download_weight = RUN_DIR/f'candidate_{RUN_ID}.pt'
shutil.copy2(CANDIDATE, download_weight)
files.download(str(download_weight)); files.download(str(internal_metrics)); files.download(str(audit_path))
print('Jangan salin candidate ke backend/weights/best.pt sebelum gold gate lolos.')

## Interpretasi

- Jika `gloves`/`shoes` di bawah 2.000 instance, dataset ini belum kuat sebagai sumber tunggal kelas tersebut.
- Internal tinggi tetapi gold rendah berarti domain mismatch/overfitting, bukan keberhasilan produksi.
- Gold reviewed tidak pernah masuk training (dicek anti-leak via SHA-256 di sel 10).
- Model aktif tetap v2 sampai candidate lolos gold dan dipromosikan melalui proses terpisah.